# Directories and Paths

*Learn to locate input files and prepare output paths with `os`, `os.path`, and a clear understanding of `sys.path`.*

Reliable file I/O starts before `open()`: a program must know its working directory, build a portable path, and keep supplied inputs separate from generated outputs.


## Before You Run

In the course JupyterLite site, choose **Add files**, select `study_plan.txt`, and enter `assets/02_text_files` for **Save folder**. The site creates missing folders and stores the file as `/home/pyodide/assets/02_text_files/study_plan.txt`.

An upload-window relative folder starts at `/home/pyodide`. A relative path inside Python starts at the current working directory set in the first code cell.


## File I/O Path Map

| Task | Tool |
|---|---|
| Check the working directory | `os.getcwd()` |
| Join path parts | `os.path.join()` |
| Convert to an absolute path | `os.path.abspath()` |
| Inspect a name or parent | `os.path.basename()`, `os.path.dirname()` |
| Check a path | `os.path.exists()`, `os.path.isfile()`, `os.path.isdir()` |
| Create and list directories | `os.makedirs()`, `os.listdir()` |
| Rename or remove an output | `os.replace()`, `os.remove()` |

`os.path` works with filesystem paths. `sys.path` has a different job: it is the list of locations Python searches when importing modules.


## Working Directory and `sys.path`

The course JupyterLite runtime starts from `/home/pyodide`. This is a virtual filesystem inside the browser, not a path on the learner's computer or the web server. `os.chdir()` sets the base before any relative data-file path is used.

Import statements search the entries in `sys.path`. That module search list is different from the working directory used to resolve relative data-file paths.


In [1]:
import os
import sys

jupyterlite_base_directory = "/home/pyodide"
chapter_name = "01-3_Python_Basics_III"
launch_directory = os.getcwd()

if os.path.isdir(jupyterlite_base_directory):
    base_directory = jupyterlite_base_directory
elif os.path.basename(launch_directory) == chapter_name:
    base_directory = launch_directory
else:
    base_directory = os.path.join(launch_directory, chapter_name)

if not os.path.isdir(base_directory):
    raise FileNotFoundError(f"Base directory not found: {base_directory}")

os.chdir(base_directory)

current_directory = os.getcwd()

print("JupyterLite base:", jupyterlite_base_directory)
print("Current directory name:", os.path.basename(current_directory))
print("Platform:", sys.platform)
print("sys.path is a list:", isinstance(sys.path, list))
print("Import search entries:", len(sys.path))


JupyterLite base: /home/pyodide
Current directory name: 01-3_Python_Basics_III
Platform: linux
sys.path is a list: True
Import search entries: 5


On the course site, the current directory name is `pyodide`. During repository validation, the same cell falls back to the chapter directory. `sys.path` is inspected only to distinguish module lookup from data-file lookup; the lesson does not modify it.


## Building and Inspecting a Path

`os.path.join()` uses the correct separator for the current operating system. The returned string describes a location; it does not create a directory or file.


In [2]:
relative_input_path = os.path.join(
    "assets",
    "02_text_files",
    "study_plan.txt",
)
parent_directory = os.path.dirname(relative_input_path)
file_name = os.path.basename(relative_input_path)
file_stem, file_extension = os.path.splitext(file_name)

print("Relative path:", relative_input_path)
print("Parent directory:", parent_directory)
print("File name:", file_name)
print("Stem:", file_stem)
print("Extension:", file_extension)


Relative path: assets/02_text_files/study_plan.txt
Parent directory: assets/02_text_files
File name: study_plan.txt
Stem: study_plan
Extension: .txt


The directory, base name, stem, and extension can be inspected separately. This is useful when selecting files or constructing related output names.


## Relative and Absolute Paths

Relative paths are portable inside a project. `os.path.abspath()` resolves a relative path from the current working directory when an absolute location is needed.


In [3]:
absolute_input_path = os.path.abspath(relative_input_path)

print("Relative path is absolute:", os.path.isabs(relative_input_path))
print("Resolved path is absolute:", os.path.isabs(absolute_input_path))
print("Resolved file name:", os.path.basename(absolute_input_path))


Relative path is absolute: False
Resolved path is absolute: True
Resolved file name: study_plan.txt


The relative and absolute strings identify the same intended file because the notebook first sets a known base directory. On JupyterLite, the resolved path begins with `/home/pyodide`; local validation uses the repository chapter directory.


## Locating the Uploaded Input

The folder entered in **Add files** becomes part of the path. Build that same numbered folder with `os.path.join()` and verify the supplied file before attempting to open it.


In [4]:
input_directory = os.path.join(
    base_directory,
    "assets",
    "02_text_files",
)
input_path = os.path.join(input_directory, "study_plan.txt")

print("Input folder:", os.path.relpath(input_directory, base_directory))
print("Input file:", os.path.basename(input_path))
print("Input exists:", os.path.isfile(input_path))

assert os.path.isfile(input_path), (
    "Add study_plan.txt and set Save folder to assets/02_text_files."
)


Input folder: assets/02_text_files
Input file: study_plan.txt
Input exists: True


The relative upload destination `assets/02_text_files` and the Python path now agree. Uploading to the same complete path replaces that file; the same filename in another folder remains separate.


## Setting Input and Output Directories

The input directory contains the uploaded example file. Generated files go in `outputs/01_directory_and_path`, where learners can inspect them in the JupyterLite file browser.


In [5]:
output_directory = os.path.join(
    base_directory,
    "outputs",
    "01_directory_and_path",
)
os.makedirs(output_directory, exist_ok=True)

print("Input file exists:", os.path.isfile(input_path))
print("Output folder:", os.path.relpath(output_directory, base_directory))
print("Output directory exists:", os.path.isdir(output_directory))


Input file exists: True
Output folder: outputs/01_directory_and_path
Output directory exists: True


The supplied input and generated output have visibly different locations. Fixed output names make repeated runs replace the same files while leaving the uploaded asset unchanged.


## Creating and Inspecting an Output File

Build the full output path before opening it. Text files use an explicit UTF-8 encoding and a `with` block so the file closes automatically.


In [6]:
output_path = os.path.join(output_directory, "path_summary.txt")
summary_text = "Input: study_plan.txt\nStatus: ready\n"

with open(output_path, mode="w", encoding="utf-8") as output_file:
    output_file.write(summary_text)

print("Output is a file:", os.path.isfile(output_path))
print("Output file name:", os.path.basename(output_path))
print("Output size in bytes:", os.path.getsize(output_path))


Output is a file: True
Output file name: path_summary.txt
Output size in bytes: 36


`os.path.isfile()` confirms the save target is a regular file, while `os.path.getsize()` reports the stored byte count.


## Listing and Filtering Files

`os.listdir()` returns names inside a directory. Sort the names for deterministic output, then use `os.path.splitext()` to select an extension.


In [7]:
input_names = sorted(os.listdir(input_directory))
text_output_names = sorted(
    name
    for name in os.listdir(output_directory)
    if os.path.splitext(name)[1] == ".txt"
)

print("Input directory:", input_names)
print("Text outputs:", text_output_names)


Input directory: ['study_plan.txt']
Text outputs: ['path_summary.txt']


The input listing contains the supplied lesson file. The output listing contains only the text file created by this notebook.


## Copying and Renaming an Output

`shutil.copy2()` creates a copy. `os.replace()` moves that program-owned copy to a new name and replaces an existing destination if necessary.


In [8]:
from shutil import copy2

copied_path = os.path.join(output_directory, "study_plan_copy.txt")
renamed_path = os.path.join(output_directory, "study_plan_backup.txt")

copy2(input_path, copied_path)
os.replace(copied_path, renamed_path)

print("Old copy exists:", os.path.exists(copied_path))
print("Renamed copy exists:", os.path.isfile(renamed_path))


Old copy exists: False
Renamed copy exists: True


The supplied asset remains unchanged. Only its copy inside the program-owned output directory is renamed.


## Removing a Program-Owned File

Delete only a file that the current program created and whose exact path is known. Never use a broad directory or unresolved pattern as a deletion target.


In [9]:
os.remove(renamed_path)

print("Program-owned copy exists:", os.path.exists(renamed_path))
print("Supplied input still exists:", os.path.isfile(input_path))


Program-owned copy exists: False
Supplied input still exists: True


Cleanup removes only the generated copy. The supplied input remains available for every later Run All.


## Handling a Missing Path

Check expected inputs before opening them and raise a precise exception that names the missing file.


In [10]:
missing_path = os.path.join(input_directory, "missing_plan.txt")

try:
    if not os.path.isfile(missing_path):
        raise FileNotFoundError(os.path.basename(missing_path))
except FileNotFoundError as error:
    print("Missing input:", error)


Missing input: missing_plan.txt


## Creating and Reopening a Python Module

A Python module is a text file whose name ends in `.py`. Save a small module in the program-owned output directory, then reopen it as text to verify the exact source code on disk.


In [11]:
module_path = os.path.join(output_directory, "study_tools.py")
module_source = (
    "def format_study_status(topic, minutes):\n"
    "    return f'{topic}: {minutes} minutes'\n"
)

with open(module_path, mode="w", encoding="utf-8") as module_file:
    module_file.write(module_source)

with open(module_path, mode="r", encoding="utf-8") as module_file:
    reopened_module_source = module_file.read()

print("Module file:", os.path.basename(module_path))
print("Source matches:", reopened_module_source == module_source)
print(reopened_module_source)


Module file: study_tools.py
Source matches: True
def format_study_status(topic, minutes):
    return f'{topic}: {minutes} minutes'



The module is ordinary UTF-8 text, so the familiar write-and-reopen verification pattern applies before Python loads it.


## Loading the Module from Its File Path

A normal `import study_tools` searches `sys.path`. Because this generated module lives outside that search path, `runpy.run_path()` executes the exact `.py` file and returns its global names without changing `sys.path`.


In [12]:
import runpy

import_path_before = list(sys.path)
module_namespace = runpy.run_path(module_path)
format_study_status = module_namespace["format_study_status"]

print("Loaded module file:", os.path.basename(module_path))
print("Function result:", format_study_status("File I/O", 45))
print("sys.path unchanged:", sys.path == import_path_before)


Loaded module file: study_tools.py
Function result: File I/O: 45 minutes
sys.path unchanged: True


Python executes the saved `.py` file and exposes its function through the returned namespace. The module-import search path remains unchanged.


## Use Cases

Common uses include locating supplied datasets, separating raw inputs from generated reports, checking file types by extension, and preparing date-organized output paths.


### Organizing Reports by Date

Multiple calls to `os.path.join()` can build a deeper location. This example constructs a path only; another I/O step would create the directories before saving.


In [13]:
report_path = os.path.join(
    "reports",
    "2026",
    "09",
    "weekly_summary.txt",
)

print("Report path:", report_path)
print("Report file name:", os.path.basename(report_path))


Report path: reports/2026/09/weekly_summary.txt
Report file name: weekly_summary.txt


Set a known working directory, keep input and output paths separate, build paths with `os.path.join()`, and create parent directories before writing. Remember that `sys.path` is for module imports. A `.py` module can be written and reopened like text, then loaded from an exact path without changing that search list.
